# Day 2: Large-Scale JSONL Data Processing for AI Datasets

## 1. Core Theory (Just-in-Time)

### The "Why"
In AI engineering, datasets are rarely small enough to fit comfortably in memory (RAM). When fine-tuning Large Language Models (LLMs), preparing Retrieval-Augmented Generation (RAG) knowledge bases, or evaluating model outputs, you will frequently encounter datasets that are gigabytes or terabytes in size. 

Loading a 50GB JSON file into memory with `json.load()` will crash most systems (Out of Memory - OOM error). The standard solution in the AI industry is **JSON Lines (JSONL)**. 

### The "How"
A JSONL file contains one valid JSON object per line. This structure allows us to:
1. **Stream the data:** Read, process, and write one line (or a small batch of lines) at a time, keeping memory usage constant regardless of file size.
2. **Parallelize processing:** Because each line is independent, it's easy to split a JSONL file across multiple CPU cores or machines.
3. **Handle corruption gracefully:** If a single line is malformed, we can catch the error, log it, and continue processing the rest of the file.

We will use modern Python features to handle this:
- **Generators (`yield`):** For lazy evaluation/streaming of data.
- **`pydantic`:** For robust, strictly-typed data validation on each line.


### AI Security Implications
When processing large datasets, particularly those scraped from the web or submitted by users, security must be built into the pipeline:
1. **PII Protection:** Data may contain Personally Identifiable Information (PII) like emails or phone numbers. A robust processing pipeline should implement redaction or filtering steps before saving to a final dataset.
2. **Fallback Mechanisms:** When parsing fails or data is corrupted, the system should log the failure and skip the record instead of crashing the entire batch process.
3. **Prompt Injection & Malicious Payloads:** Even at the dataset level, expect malicious inputs. Sanitize text fields to prevent downstream issues when this data is fed into an LLM.


## 2. Code Implementation

We will progress from a basic streaming example to a full, production-grade object-oriented system.

### Basic Implementation
Isolates the core concept of generator-based streaming with minimal boilerplate.

In [1]:
import json
from pathlib import Path

def basic_jsonl_reader(filepath: Path):
    """A minimal generator to read a JSONL file line-by-line."""
    with filepath.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)

# Create a small dummy file
dummy_path = Path("basic_dummy.jsonl")
with dummy_path.open("w", encoding="utf-8") as f:
    f.write('{"id": 1, "text": "Hello world"}\n')
    f.write('{"id": 2, "text": "AI Engineering"}\n')

# Process it
print("Basic Implementation Output:")
for record in basic_jsonl_reader(dummy_path):
    print(record)

# Cleanup
dummy_path.unlink()

Basic Implementation Output:
{'id': 1, 'text': 'Hello world'}
{'id': 2, 'text': 'AI Engineering'}


### Medium Implementation
Emphasizes clean OOP, state management, and strict data validation using `pydantic`.

In [2]:
import json
import logging
from typing import Iterator, Optional
from pydantic import BaseModel, ValidationError, Field
from pathlib import Path

# Setup basic logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

class DatasetRecord(BaseModel):
    """Schema for a single document in our AI dataset."""
    id: str
    text: str = Field(..., min_length=1)
    metadata: Optional[dict] = Field(default_factory=dict)

class JSONLProcessor:
    """Manages the reading and validation of JSONL files."""
    def __init__(self, filepath: Path):
        self.filepath = filepath
        self.processed_count = 0
        self.error_count = 0

    def process_stream(self) -> Iterator[DatasetRecord]:
        """Streams and validates records."""
        with self.filepath.open("r", encoding="utf-8") as f:
            for line_number, line in enumerate(f, start=1):
                line = line.strip()
                if not line:
                    continue
                try:
                    raw_dict = json.loads(line)
                    record = DatasetRecord(**raw_dict)
                    self.processed_count += 1
                    yield record
                except (json.JSONDecodeError, ValidationError) as e:
                    self.error_count += 1
                    logging.warning(f"Error at line {line_number}: {e}")

# Test the medium implementation
med_path = Path("med_dummy.jsonl")
with med_path.open("w", encoding="utf-8") as f:
    f.write('{"id": "doc1", "text": "Valid text", "metadata": {}}\n')
    f.write('{"id": "doc2", "text": ""} \n') # Invalid: empty text

processor = JSONLProcessor(med_path)
print("\nMedium Implementation Output:")
for doc in processor.process_stream():
    print(doc.model_dump_json())

print(f"Stats - Processed: {processor.processed_count}, Errors: {processor.error_count}")
med_path.unlink()

text
  String should have at least 1 character [type=string_too_short, input_value='', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/string_too_short



Medium Implementation Output:
{"id":"doc1","text":"Valid text","metadata":{}}
Stats - Processed: 1, Errors: 1


### Advanced Implementation
A production-grade pipeline demonstrating strict type hinting, batch processing for downstream API usage, exact import syntax, error handling, and AI security measures like basic PII redaction and robust fallback mechanisms.

In [3]:
import json
import logging
import re
from typing import Iterator, List, Optional
from pathlib import Path
from pydantic import BaseModel, ValidationError, Field

logger = logging.getLogger("advanced_pipeline")
logger.setLevel(logging.INFO)
if not logger.handlers:
    ch = logging.StreamHandler()
    ch.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))
    logger.addHandler(ch)

class SecureDocument(BaseModel):
    """Production schema ensuring robust validation and default fallbacks."""
    id: str = Field(..., description="Unique identifier for the document")
    text: str = Field(..., min_length=1, description="The content of the document")
    source: str = Field(default="unknown", description="Data source fallback")
    is_safe: bool = Field(default=True, description="Flag for malicious content")

class SecureDataPipeline:
    """Production-grade data processor with PII filtering and batching."""
    
    # Simple regex for basic PII masking (e.g., standard email format)
    EMAIL_REGEX = re.compile(r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+")
    
    def __init__(self, input_path: Path, output_path: Path, batch_size: int = 100):
        self.input_path = input_path
        self.output_path = output_path
        self.batch_size = batch_size
        self.stats = {"processed": 0, "errors": 0, "redacted": 0, "written": 0}

    def _sanitize_text(self, text: str) -> str:
        """Applies basic PII protection by masking emails."""
        sanitized, count = self.EMAIL_REGEX.subn("[REDACTED_EMAIL]", text)
        if count > 0:
            self.stats["redacted"] += count
        return sanitized

    def stream_and_validate(self) -> Iterator[SecureDocument]:
        """Streams records, sanitizes data, and yields valid documents."""
        if not self.input_path.exists():
            raise FileNotFoundError(f"Missing dataset at {self.input_path}")

        with self.input_path.open("r", encoding="utf-8") as f:
            for line_num, line in enumerate(f, start=1):
                line = line.strip()
                if not line:
                    continue
                try:
                    raw_dict = json.loads(line)
                    # Apply AI Security: PII masking on text fields
                    if "text" in raw_dict and isinstance(raw_dict["text"], str):
                        raw_dict["text"] = self._sanitize_text(raw_dict["text"])
                        
                    doc = SecureDocument(**raw_dict)
                    self.stats["processed"] += 1
                    yield doc
                except (json.JSONDecodeError, ValidationError) as e:
                    self.stats["errors"] += 1
                    logger.debug(f"Validation failure at line {line_num}: {e}")
                    # Fallback Mechanism: Skip the bad record instead of crashing

    def run_batch_pipeline(self) -> None:
        """Runs the pipeline, grouping valid records into batches for writing."""
        batch: List[SecureDocument] = []
        
        # Ensure output directory exists
        self.output_path.parent.mkdir(parents=True, exist_ok=True)
        
        with self.output_path.open("w", encoding="utf-8") as out_f:
            for doc in self.stream_and_validate():
                batch.append(doc)
                
                if len(batch) >= self.batch_size:
                    self._write_batch(batch, out_f)
                    batch = []
                    
            if batch:
                self._write_batch(batch, out_f)
                
        logger.info(f"Pipeline complete. Stats: {self.stats}")

    def _write_batch(self, batch: List[SecureDocument], file_obj) -> None:
        """Writes a batch of validated records to disk safely."""
        for doc in batch:
            file_obj.write(doc.model_dump_json() + "\n")
            self.stats["written"] += 1

# --- Execution ---
if __name__ == "__main__":
    adv_in = Path("adv_input.jsonl")
    adv_out = Path("adv_output.jsonl")
    
    # Generate mock production data
    with adv_in.open("w", encoding="utf-8") as f:
        f.write('{"id": "p1", "text": "Contact user at admin@example.com for info.", "source": "web"}\n')
        f.write('{"id": "p2", "text": "This is safe data.", "source": "api"}\n')
        f.write('{"id": "p3", "text": "", "source": "web"} \n') # Should error and fallback
    
    pipeline = SecureDataPipeline(input_path=adv_in, output_path=adv_out, batch_size=2)
    pipeline.run_batch_pipeline()
    
    # Cleanup
    adv_in.unlink()
    adv_out.unlink()


2026-08-20 12:51:47,908 [INFO] Pipeline complete. Stats: {'processed': 2, 'errors': 1, 'redacted': 1, 'written': 2}


INFO: Pipeline complete. Stats: {'processed': 2, 'errors': 1, 'redacted': 1, 'written': 2}


## 3. Common Pitfalls in Production

1. **Ignoring Encoding:** Always explicitly set `encoding="utf-8"` when opening files (`open(filepath, "r", encoding="utf-8")`). Systems default to different encodings (e.g., Windows defaults to cp1252), which will crash when encountering emojis or special characters common in web-scraped AI datasets.
2. **`json.loads()` vs `json.load()`:** 
   - `json.load(f)` reads the *entire* file object at once. Do not use this for large JSON files.
   - `json.loads(string)` parses a *string*. In JSONL processing, we use `json.loads(line)` inside a loop.
3. **Failing to validate inputs:** Upstream data pipelines frequently change. If you assume a `"metadata"` field is always a dictionary and suddenly it's `null`, your downstream embedding script will throw an unhandled `TypeError` three hours into a ten-hour job. Always use a validator like `pydantic`.
4. **Missing batch processing:** Sending data to APIs (like LLM endpoints or Vector DBs) one by one is incredibly slow due to network latency. Batching (as shown in the code) allows you to utilize batch-API endpoints for massive speedups.
5. **No Fallback Mechanisms:** Crashing a 5-hour batch job at 99% completion because of one malformed record is a common rookie mistake. Always handle individual record exceptions gracefully.

## 4. Practical Lab / Homework

**Task:** Build a secure dataset filter for a fine-tuning pipeline.

1. **Setup:** Programmatically generate a dataset file called `lab_input.jsonl` containing 50 records. Include fields: `id`, `text`, and `quality_score` (a float between 0.0 and 1.0). In some records, include dummy PII (like phone numbers or emails) and intentionally malformed JSON to test your fallbacks.
2. **Filter & Transform:** Write a script that reads this JSONL file stream using the batching technique. 
3. **Validation & Security:** Use `pydantic` to ensure `quality_score` is a float. Implement a basic redaction method to mask sensitive data in the `text` field.
4. **Write:** If a record has a `quality_score` >= 0.8 and parses successfully, write it out to a new file called `high_quality_dataset.jsonl`.
5. **Constraint:** You must process the file line-by-line and write line-by-line. The entire dataset must never be in memory at the same time.

Use the cell below to implement your solution. When finished, record a brief async video walkthrough of your design decisions emphasizing your object-oriented approach and fallback strategy.

In [4]:
from pathlib import Path
from typing import Iterator, List
from pydantic import BaseModel, Field, ValidationError
import json
import re

# 1. Define your Pydantic schema
class TrainingRecord(BaseModel):
    id: str
    text: str
    quality_score: float = Field(..., ge=0.0, le=1.0)

class LabPipeline:
    def __init__(self, in_file: Path, out_file: Path, batch_size: int = 10):
        self.in_file = in_file
        self.out_file = out_file
        self.batch_size = batch_size
        self.email_regex = re.compile(r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+")
        
    def sanitize(self, text: str) -> str:
        return self.email_regex.sub("[REDACTED]", text)
        
    def stream_filtered(self) -> Iterator[TrainingRecord]:
        with self.in_file.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    data = json.loads(line)
                    if "text" in data and isinstance(data["text"], str):
                        data["text"] = self.sanitize(data["text"])
                    record = TrainingRecord(**data)
                    if record.quality_score >= 0.8:
                        yield record
                except (json.JSONDecodeError, ValidationError):
                    # Fallback: skip invalid
                    continue

    def run(self):
        batch = []
        with self.out_file.open("w", encoding="utf-8") as f_out:
            for record in self.stream_filtered():
                batch.append(record)
                if len(batch) >= self.batch_size:
                    for b in batch:
                        f_out.write(b.model_dump_json() + "\n")
                    batch = []
            for b in batch:
                f_out.write(b.model_dump_json() + "\n")

# Execution setup
if __name__ == "__main__":
    lab_in = Path("lab_input.jsonl")
    lab_out = Path("high_quality_dataset.jsonl")
    
    # Generate dummy data
    with lab_in.open("w", encoding="utf-8") as f:
        f.write('{"id": "1", "text": "Good data no pii", "quality_score": 0.9}\n')
        f.write('{"id": "2", "text": "Contact me at test@test.com", "quality_score": 0.95}\n')
        f.write('{"id": "3", "text": "Bad data", "quality_score": 0.5}\n')
        f.write('{"id": "4", "text": "Malformed JSON", "quality_score": }\n')
        
    pipeline = LabPipeline(lab_in, lab_out)
    pipeline.run()
    
    # Cleanup
    if lab_in.exists(): lab_in.unlink()
    if lab_out.exists(): lab_out.unlink()


## 5. Reference Links
- [JSON Lines Documentation](https://jsonlines.org/)
- [Pydantic V2 Models](https://docs.pydantic.dev/latest/concepts/models/)
- [Python `json` library](https://docs.python.org/3/library/json.html)